# Micro-Reformer Hydrogen Production Analysis
This Jupyter Notebook runs the core Python models (`kinetics_model.py` and `mass_energy_balance.py`) to simulate the micro-reformer chemistry and plots the results directly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from kinetics_model import run_pfr
from mass_energy_balance import MicroReformerBalance

# Set professional plotting style
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    pass
plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})


## 1. Thermodynamic & Kinetic Sweep: Effect of Temperature
We sweep the temperature from 500°C to 900°C to observe how Methane Conversion and Hydrogen Yield change under 1 bar and Steam/Carbon (S/C) = 3.


In [ ]:
temps_C = np.linspace(500, 900, 15)
temps_K = temps_C + 273.15
conversions = []
yields = []
selectivities = []

for T in temps_K:
    # Run the Xu & Froment PFR solver
    res = run_pfr(T_K=T, P_bar=1.0, SC_ratio=3.0, W_cat=0.5)
    conversions.append(res['X_CH4'][-1])
    yields.append(res['Y_H2'][-1])
    selectivities.append(res['S_CO'][-1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Conversion and Yield
ax1.plot(temps_C, conversions, label='CH$_4$ Conversion (%)', color='#ef4444', marker='o', lw=2)
ax1.plot(temps_C, yields, label='H$_2$ Yield (%)', color='#10b981', marker='s', lw=2)
ax1.set_title('Methane Conversion & Hydrogen Yield')
ax1.set_xlabel('Temperature [°C]')
ax1.set_ylabel('Percentage [%]')
ax1.legend()

# Plot 2: CO Selectivity
ax2.plot(temps_C, selectivities, label='CO Selectivity (%)', color='#f59e0b', marker='^', lw=2)
ax2.set_title('Carbon Monoxide Selectivity (WGS limit)')
ax2.set_xlabel('Temperature [°C]')
ax2.set_ylabel('Selectivity [%]')
ax2.legend()

plt.tight_layout()
plt.show()


## 2. Mass & Energy Balance
We instantiate the rigorous `MicroReformerBalance` class at the optimal condition ($800°\text{C}$). This performs strict stoichiometric unit testing and calculates the exact heat duty using Kirchhoff's thermodynamic law.


In [ ]:
# Nominal operating condition
T_opt_K = 800 + 273.15
balance = MicroReformerBalance(T_K=T_opt_K, P_bar=1.0, SC_ratio=3.0, F_CH4_in=1e-4)
result = balance.run()

print("="*40)
print("  Micro-Reformer Performance Profile  ")
print("="*40)
print(f"Operating Temp: 800 °C")
print(f"Steam/Carbon:   3.0")
print("-"*40)
print(f"CH4 Conversion: {result.X_CH4:.2f} %")
print(f"H2 Yield:       {result.Y_H2:.2f} %")
print(f"Thermal Duty:   {result.Q_req_W:.2f} W  (Endothermic)")
print(f"Thermal Eff:    {result.eta_thermal:.2f} %")
print("="*40)

# Verify Conservation of Mass
atm = balance.atom_balance_check(result)
print(f"\nAtom Balance Errors: C: {atm['C_err_%']:.6f}%, H: {atm['H_err_%']:.6f}%, O: {atm['O_err_%']:.6f}%")
if atm['C_err_%'] < 0.01:
    print("\u2705 Conservation of mass verified.")
